In [1]:
import os
import re
import pickle
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Bidirectional,
    GRU,
    Dense,
    Dropout,
    Attention,
    GlobalAveragePooling1D
)

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint
)

# ===================================
# CREATE MODELS DIRECTORY
# ===================================

os.makedirs("models", exist_ok=True)

# ===================================
# LOAD DATA
# ===================================

print("Loading dataset...")

fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

fake["label"] = 1
true["label"] = 0

df = pd.concat([fake, true], ignore_index=True)

df = df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# ===================================
# CLEAN TEXT
# ===================================

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z ]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")

df["content"] = (
    df["title"] + " " + df["text"]
)

df["content"] = df["content"].apply(clean_text)

# ===================================
# TOKENIZATION
# ===================================

MAX_WORDS = 50000
MAX_LEN = 500

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(df["content"])

X = tokenizer.texts_to_sequences(
    df["content"]
)

X = pad_sequences(
    X,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

y = df["label"].values

# ===================================
# TRAIN TEST SPLIT
# ===================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ===================================
# MODEL
# ===================================

inputs = Input(shape=(MAX_LEN,))

embedding = Embedding(
    input_dim=MAX_WORDS,
    output_dim=128
)(inputs)

gru1 = Bidirectional(
    GRU(
        128,
        return_sequences=True,
        dropout=0.2
    )
)(embedding)

gru2 = Bidirectional(
    GRU(
        64,
        return_sequences=True,
        dropout=0.2
    )
)(gru1)

attention = Attention()(
    [gru2, gru2]
)

pool = GlobalAveragePooling1D()(
    attention
)

dense = Dense(
    64,
    activation="relu"
)(pool)

drop = Dropout(0.3)(
    dense
)

outputs = Dense(
    1,
    activation="sigmoid"
)(drop)

model = Model(
    inputs=inputs,
    outputs=outputs
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ===================================
# CALLBACKS
# ===================================

checkpoint = ModelCheckpoint(
  "models/bigru_attention_model.keras",
  monitor="val_accuracy",
  mode="max",
  save_best_only=True,
  verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

# ===================================
# TRAIN
# ===================================

history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=2,
    batch_size=64,
    callbacks=[
        checkpoint,
        early_stop
    ]
)

# ===================================
# EVALUATE
# ===================================

preds = (
    model.predict(X_test) > 0.5
).astype(int)

print(
    classification_report(
        y_test,
        preds
    )
)

# ===================================
# SAVE FINAL MODEL
# ===================================

model.save(
    "models/final_bigru_attention.keras"
)

# ===================================
# SAVE TOKENIZER
# ===================================

with open(
    "models/tokenizer.pkl",
    "wb"
) as f:

    pickle.dump(
        tokenizer,
        f
    )

print("\nTraining Complete")
print("Saved:")
print("models/bigru_attention_model.keras")
print("models/final_bigru_attention.keras")
print("models/tokenizer.pkl")

Loading dataset...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 500)               │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 500, 128)          │       6,400,000 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional (Bidirectional) │ (None, 500, 256)          │         198,144 │ embedding[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bidirectional_1               │ (None, 500, 128)          │         123,648 │ bidirectional[0][0]        │
│ (Bidirectional)               │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention (Attention)         │ (None, 500, 128)          │               0 │ bidirectional_1[0][0],     │
│                               │                           │                 │ bidirectional_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ global_average_pooling1d      │ (None, 128)               │               0 │ attention[0][0]            │
│ (GlobalAveragePooling1D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense (Dense)                 │ (None, 64)                │           8,256 │ global_average_pooling1d[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout (Dropout)             │ (None, 64)                │               0 │ dense[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_1 (Dense)               │ (None, 1)                 │              65 │ dropout[0][0]              │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 6,730,113 (25.67 MB)

 Trainable params: 6,730,113 (25.67 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/2
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 8s/step - accuracy: 0.8567 - loss: 0.2712
Epoch 1: val_accuracy improved from None to 0.99791, saving model to models/bigru_attention_model.keras

Epoch 1: finished saving model to models/bigru_attention_model.keras
449/449 ━━━━━━━━━━━━━━━━━━━━ 3505s 8s/step - accuracy: 0.9494 - loss: 0.1119 - val_accuracy: 0.9979 - val_loss: 0.0108
Epoch 2/2
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.9981 - loss: 0.0099
Epoch 2: val_accuracy did not improve from 0.99791
449/449 ━━━━━━━━━━━━━━━━━━━━ 4192s 9s/step - accuracy: 0.9976 - loss: 0.0120 - val_accuracy: 0.9896 - val_loss: 0.0495
Restoring model weights from the end of the best epoch: 1.
281/281 ━━━━━━━━━━━━━━━━━━━━ 225s 790ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4284
           1       1.00      1.00      1.00      4696

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00    